---
title: "Chapter 26: Data schema validation with Pandera"
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sambaiga/ds-mlops-path/blob/main/tutorials/02-dev-tools/26-pandera-schema-validation.ipynb) [![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue.svg?logo=jupyter&logoColor=white)](https://raw.githubusercontent.com/sambaiga/ds-mlops-path/main/tutorials/02-dev-tools/26-pandera-schema-validation.ipynb)

The grade-predictor pipeline finishes training. Accuracy: 87%. You deploy it.

Three weeks later a data engineer flags something: 23 rows in `university_analytics.csv` have `final_score = 999`. The data entry system used 999 as a sentinel for "not recorded". Nobody documented it. The model trained on those 23 rows as if 999 were a valid exam score. There was no error, no warning. The model just learned the wrong thing for 23 students.

The check you ran at load time called `.dtypes` and printed `.head()`. Neither caught a score of 999. Validation by inspection runs once, in a notebook cell, by whoever is at the keyboard. It is forgotten the next time the pipeline runs.

What you needed was a **data contract**: a formal specification of what the DataFrame must look like: column types, allowed value ranges, uniqueness constraints, that runs automatically every time data enters the pipeline and fails loudly with a precise error when violated. Pydantic gave you this for individual records at the system boundary. Pandera gives you this for whole DataFrames inside the pipeline.

**Next:** [Chapter 27: Logging](27-logging.ipynb) adds the audit trail that records which validation errors occurred and when, the permanent record that Pandera's exceptions do not keep by themselves.

[![View Source on GitHub](https://img.shields.io/badge/Source-GitHub-181717.svg?logo=github)](https://github.com/sambaiga/ds-mlops-path/blob/main/tutorials/02-dev-tools/26-pandera-schema-validation.ipynb)

::: {.callout-note collapse="true" icon=false}
## Learning objectives

By the end of this chapter you will be able to:

| # | Skill | Covered in |
| --- | --- | --- |
| 1 | Explain the difference between row-level (Pydantic) and schema-level (Pandera) validation | Sec. 1 |
| 2 | Define a Pandera `DataFrameSchema` with column types and constraints | Sec. 2 |
| 3 | Use the class-based API with `pa.DataFrameModel` for typed schemas | Sec. 3 |
| 4 | Write custom element-wise and series-level checks | Sec. 4 |
| 5 | Validate DataFrames in a pipeline and collect errors without stopping | Sec. 5 |
| 6 | Use Pandera schemas as pytest fixtures to document data contracts | Sec. 6 |
:::

Pydantic validated the shape of a single record as it entered the system: one row, one pass or fail. A DataFrame is not a record. It is a table. The properties that matter at the table level are different: is `student_id` unique across all 2,400 rows? Is the distribution of `program` values consistent with what the source system should produce? Does `final_score` stay between 0 and 100 for every row, not just the one you checked manually?

These are questions Pydantic cannot answer, because Pydantic operates on one object at a time.

| | Pydantic `BaseModel` | Pandera `DataFrameSchema` |
| --- | --- | --- |
| Unit | One record (row) | Whole DataFrame |
| Checks | Type coercion, field constraints, cross-field | Column dtype, nullability, uniqueness, value ranges, statistical bounds |
| Returns | Validated model instance | Validated DataFrame |
| Error info | Field path + message | Row index + column + failed check |
| Best for | API inputs, config objects | CSVs, pipeline data, feature tables |

<div class='ark-concept'>
<span class='ark-concept-title'><i class="bi bi-key-fill"></i> Key concept: a schema is documentation that runs</span><br><br>
A Pandera schema is not a test you write once. It is a living specification: the schema says what the DataFrame must look like, and the pipeline is only allowed to proceed when the data keeps that promise. Every time the pipeline runs, the contract is checked. When the data changes in a way that violates it, you know immediately, with a row index, a column name, and the rule that failed.
</div>

The diagram below shows the contract in action: a raw DataFrame enters Pandera, the schema checks each column's type, nullability, range, and uniqueness, and either returns a validated frame or raises a `SchemaError` with the exact row index and rule that failed.

![Pandera as a schema contract for DataFrames: a raw DataFrame with mixed types and out-of-range values passes through the Pandera schema, which either returns a validated frame with only conforming rows or raises a SchemaError naming the column, row, and violated rule.](figs/pandera-contract.svg){fig-alt="Three columns: raw DataFrame with highlighted bad rows (None, 102, empty string); Pandera schema in the centre listing column rules; validated frame on the right containing only conforming rows. A red SchemaError arrow exits downward from the schema when data fails."}

## 1. Defining a `DataFrameSchema`

The functional API creates a schema by passing a dict of column names to `pa.Column` definitions. Each `pa.Column` declares the dtype and the checks it must satisfy.

In [ ]:
import numpy as np
import pandas as pd
import pandera as pa

Set `coerce=True` to let Pandera convert `"78.5"` to `78.5` before checking a `float` column. This mirrors Pydantic's type coercion at the row level. The top-level `checks` list accepts table-level constraints such as uniqueness across the entire column.

In [ ]:
schema = pa.DataFrameSchema(
    {
        "student_id": pa.Column(str, pa.Check.str_matches(r"^S\d{4}$")),
        "midterm_score": pa.Column(float, [pa.Check.ge(0.0), pa.Check.le(100.0)]),
        "final_score": pa.Column(float, [pa.Check.ge(0.0), pa.Check.le(100.0)]),
        "project_score": pa.Column(float, [pa.Check.ge(0.0), pa.Check.le(100.0)]),
        "program": pa.Column(str, pa.Check.isin(["CS", "EE", "Math", "Physics", "Biology"])),
        "has_internet": pa.Column(bool),
        "school_id": pa.Column(str, nullable=True),
        "teacher_count": pa.Column(int, pa.Check.ge(1)),
        "school_size": pa.Column(str, pa.Check.isin(["Small", "Medium", "Large"])),
        "pass_threshold": pa.Column(float, pa.Check.between(50.0, 80.0), nullable=True),
    },
    checks=[
        pa.Check(lambda df: df["student_id"].is_unique, error="student_id must be unique"),
    ],
    coerce=True,
)

print("Schema created with", len(schema.columns), "columns")

Build a small valid sample to test against:

In [ ]:
sample_df = pd.DataFrame(
    {
        "student_id": ["S0001", "S0002", "S0003"],
        "midterm_score": [78.5, 65.0, 90.0],
        "final_score": [82.0, 70.0, 88.0],
        "project_score": [91.0, 75.0, 85.0],
        "program": ["CS", "EE", "Math"],
        "has_internet": [True, False, True],
        "school_id": ["SCH01", "SCH01", "SCH02"],
        "teacher_count": [12, 8, 15],
        "school_size": ["Large", "Medium", "Large"],
        "pass_threshold": [60.0, 60.0, 65.0],
    }
)
sample_df.head()

Call `.validate()` on a conforming DataFrame. It returns the validated frame unchanged:

In [ ]:
validated = schema.validate(sample_df)
print("Validation passed:", validated.shape)

Introduce a row with `midterm_score = 150` and confirm Pandera raises `SchemaError` with the row index and the violated rule:

In [ ]:
bad_df = sample_df.copy()
bad_df.loc[0, "midterm_score"] = 150.0

try:
    schema.validate(bad_df)
except pa.errors.SchemaError as e:
    print(type(e).__name__)
    print(e)

<div class='ark-activity'>
<span class='ark-activity-title'><i class="bi bi-puzzle-fill"></i> Activity 1: duplicate student_id</span><br><br>
Create a DataFrame where two rows share the same <code>student_id</code>. Run <code>schema.validate(df)</code> and confirm it raises a <code>SchemaError</code> with a message about uniqueness.
<pre>dup_df = sample_df.copy()
dup_df.loc[1, "student_id"] = "S0001"  # duplicate
schema.validate(dup_df)  # should raise</pre>
</div>

## 2. `DataFrameModel`: the class-based API

The functional `DataFrameSchema` API is explicit but verbose. Pandera's class-based API mirrors Pydantic's `BaseModel`: columns become annotated class fields, and `pa.Field` carries the constraints.

In [ ]:
from typing import Optional

import pandera as pa
from pandera.typing import Series

Define the schema as a class. The `Config` inner class sets options such as `coerce`. A `@pa.check` classmethod adds a column-level constraint that the built-in `pa.Field` arguments cannot express:

In [ ]:
class StudentDataSchema(pa.DataFrameModel):
    student_id: Series[str] = pa.Field(str_matches=r"^S\d{4}$")
    midterm_score: Series[float] = pa.Field(ge=0.0, le=100.0)
    final_score: Series[float] = pa.Field(ge=0.0, le=100.0)
    project_score: Series[float] = pa.Field(ge=0.0, le=100.0)
    program: Series[str] = pa.Field(isin=["CS", "EE", "Math", "Physics", "Biology"])
    has_internet: Series[bool]
    school_id: Series[str] | None = pa.Field(nullable=True)
    teacher_count: Series[int] = pa.Field(ge=1)
    school_size: Series[str] = pa.Field(isin=["Small", "Medium", "Large"])

    class Config:
        coerce = True

    @pa.check("student_id")
    def student_id_unique(cls, series: Series[str]) -> bool:  # noqa: N805
        return series.is_unique

Validate by calling the class name directly. No schema object needed:

In [ ]:
validated = StudentDataSchema.validate(sample_df)
print("Class-based validation passed:", validated.shape)

The `@pa.check` decorator attaches a column-level check as a classmethod. The check receives the entire `Series` and must return a boolean (or a boolean Series for element-wise checks).

<div class='ark-tip'>
<span class='ark-tip-title'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Use <code>DataFrameModel</code> for reuse, <code>DataFrameSchema</code> for quick scripts</span><br><br>
<code>DataFrameModel</code> is easier to subclass, document, and test: it reads like a dataclass and fits naturally alongside Pydantic models. <code>DataFrameSchema</code> is useful when you want to build a schema programmatically at runtime, e.g., from a config file or database metadata.
</div>

<div class='ark-activity'>
<span class='ark-activity-title'><i class="bi bi-puzzle-fill"></i> Activity 2: extend the Schema</span><br><br>
Subclass <code>StudentDataSchema</code> and add a <code>semester</code> column constrained to <code>["Fall", "Spring", "Summer"]</code>. Validate a DataFrame that includes the column with valid values, then one that has an invalid value. Confirm only the second raises a <code>SchemaError</code>.
<pre>class ExtendedSchema(StudentDataSchema):
    semester: Series[str] = pa.Field(isin=["Fall", "Spring", "Summer"])</pre>
</div>

## 3. Custom checks

Built-in checks cover ranges, null counts, string patterns, and allowed values. Custom checks handle business rules that require more logic.

In [ ]:
import pandera as pa
from pandera.typing import Series

`@pa.check("column_name")` adds a column-level check: the method receives the full `Series` and must return `True` (whole series passes) or a boolean `Series` (element-wise). `@pa.dataframe_check` receives the full DataFrame for cross-column constraints:

In [ ]:
class GradeSchema(pa.DataFrameModel):
    midterm_score: Series[float] = pa.Field(ge=0.0, le=100.0)
    final_score: Series[float] = pa.Field(ge=0.0, le=100.0)
    project_score: Series[float] = pa.Field(ge=0.0, le=100.0)

    @pa.check("midterm_score", name="not_all_perfect")
    def not_all_perfect_scores(cls, series: Series[float]) -> bool:  # noqa: N805
        return (series == 100.0).mean() < 0.5

    @pa.dataframe_check
    def weighted_average_reasonable(cls, df: pd.DataFrame) -> bool:  # noqa: N805
        avg = df["midterm_score"] * 0.3 + df["final_score"] * 0.45 + df["project_score"] * 0.25
        return (avg >= 0.0).all() and (avg <= 100.0).all()

Run it against the sample to confirm both checks pass:

In [ ]:
result = GradeSchema.validate(sample_df[["midterm_score", "final_score", "project_score"]])
print("Custom checks passed:", result.shape)

`@pa.check("column_name")` adds a column-level check: return `True` (the whole series passes), a boolean `Series` (element-wise result), or raise with a message.

`@pa.dataframe_check` gets the whole DataFrame: use it for cross-column constraints.

<div class='ark-activity'>
<span class='ark-activity-title'><i class="bi bi-puzzle-fill"></i> Activity 3: cross-Column Check</span><br><br>
Add a <code>@pa.dataframe_check</code> to <code>GradeSchema</code> that verifies <code>midterm_score</code> and <code>final_score</code> are not both 0 for the same student (a student with both scores at 0 is almost certainly an error, not a result). Confirm it passes on valid data and fails when you introduce a row with both at 0.
<pre>@pa.dataframe_check
def not_both_zero(cls, df: pd.DataFrame) -> pd.Series:
    return ~((df["midterm_score"] == 0) & (df["final_score"] == 0))</pre>
</div>

## 4. Validation in a pipeline

In a pipeline, failing on the first error stops the run and loses all subsequent error information. Use `lazy=True` to collect every failure before raising:

In [ ]:
import pandas as pd
import pandera as pa

`load_and_validate` wraps `schema.validate(df, lazy=True)`: it catches `SchemaErrors`, extracts the failure cases, drops the invalid rows, and returns the clean frame alongside a list of error dicts for logging:

In [ ]:
def load_and_validate(
    path: str,
    schema: type[pa.DataFrameModel],
    *,
    lazy: bool = True,
) -> tuple[pd.DataFrame, list[dict]]:
    df = pd.read_csv(path)

    if not lazy:
        return schema.validate(df, lazy=False), []

    try:
        return schema.validate(df, lazy=True), []
    except pa.errors.SchemaErrors as e:
        error_df = e.failure_cases
        errors = error_df.to_dict(orient="records")
        bad_idx = set(error_df["index"].dropna().astype(int).tolist())
        clean_df = df.drop(index=list(bad_idx)).reset_index(drop=True)
        return clean_df, errors

Call it with the `StudentDataSchema` defined in section 2:

In [ ]:
try:
    clean, errors = load_and_validate(
        "tutorials/data/university_analytics.csv",
        StudentDataSchema,
    )
    print(f"Clean rows: {len(clean)}, Errors: {len(errors)}")
    if errors:
        print("First error:", errors[0])
except FileNotFoundError:
    print("CSV not found in this context; run from repo root")

<div class='ark-concept'>
<span class='ark-concept-title'><i class="bi bi-key-fill"></i> Key Concept: lazy=True collects all errors; lazy=False stops on the first</span><br><br>
<code>schema.validate(df, lazy=False)</code> (the default) raises a <code>SchemaError</code> on the first failure: fast and clear for development. <code>schema.validate(df, lazy=True)</code> collects every failure and raises a <code>SchemaErrors</code> (note the plural) at the end, better for production, where you want a full error report rather than a partial run.
</div>

<div class='ark-activity'>
<span class='ark-activity-title'><i class="bi bi-puzzle-fill"></i> Activity 4: full Error Report</span><br><br>
Introduce three invalid rows into <code>sample_df</code>: one with <code>midterm_score=150.0</code>, one with an invalid <code>program</code>, and one with a duplicate <code>student_id</code>. Call <code>StudentDataSchema.validate(bad_df, lazy=True)</code>. Catch the <code>SchemaErrors</code> exception and print the <code>failure_cases</code> DataFrame showing all three failures at once.
</div>

## 5. Schemas as data contracts in tests

A Pandera schema is documentation that runs. Put it in a pytest fixture and every test that touches a DataFrame automatically validates the contract.

Define `StudentDataSchema` in `grade_predictor/schemas.py` so it can be imported by both the pipeline and the tests:

In [ ]:
# grade_predictor/schemas.py
import pandera as pa
from pandera.typing import Series


class StudentDataSchema(pa.DataFrameModel):
    student_id: Series[str] = pa.Field(str_matches=r"^S\d{4}$")
    midterm_score: Series[float] = pa.Field(ge=0.0, le=100.0)
    final_score: Series[float] = pa.Field(ge=0.0, le=100.0)
    project_score: Series[float] = pa.Field(ge=0.0, le=100.0)
    program: Series[str] = pa.Field(isin=["CS", "EE", "Math", "Physics", "Biology"])
    has_internet: Series[bool]
    teacher_count: Series[int] = pa.Field(ge=1)
    school_size: Series[str] = pa.Field(isin=["Small", "Medium", "Large"])

    class Config:
        coerce = True

A `@pytest.fixture` builds a minimal valid DataFrame. Each test gets a fresh copy:

In [ ]:
import pandas as pd
import pytest


@pytest.fixture
def valid_df():
    return pd.DataFrame(
        {
            "student_id": ["S0001", "S0002"],
            "midterm_score": [78.5, 65.0],
            "final_score": [82.0, 70.0],
            "project_score": [91.0, 75.0],
            "program": ["CS", "EE"],
            "has_internet": [True, False],
            "teacher_count": [12, 8],
            "school_size": ["Large", "Medium"],
        }
    )

Test that clean data passes the schema:

In [ ]:
def test_valid_data_passes_schema(valid_df):
    validated = StudentDataSchema.validate(valid_df)
    assert len(validated) == 2  # noqa: S101

Test that a score above 100 raises `SchemaError`:

In [ ]:
def test_invalid_score_raises(valid_df):
    bad = valid_df.copy()
    bad.loc[0, "midterm_score"] = 150.0
    with pytest.raises(pa.errors.SchemaError):
        StudentDataSchema.validate(bad)

Test that an unknown program value raises `SchemaError`:

In [ ]:
def test_invalid_program_raises(valid_df):
    bad = valid_df.copy()
    bad.loc[0, "program"] = "Underwater Basket Weaving"
    with pytest.raises(pa.errors.SchemaError):
        StudentDataSchema.validate(bad)

Run the tests from the repo root:

In [ ]:
# Save the cells above to tests/test_data_contracts.py, then:
print("Run with: uv run pytest tests/test_data_contracts.py -v")

<div class='ark-tip'>
<span class='ark-tip-title'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Use <code>pa.DataFrameModel.example()</code> to generate test data automatically</span><br><br>
Pandera can synthesise valid DataFrames from a schema: <code>StudentDataSchema.example(size=50)</code> generates 50 valid rows matching all constraints. This removes the need to hand-craft test fixtures for every new schema.
</div>

<div class='ark-activity'>
<span class='ark-activity-title'><i class="bi bi-puzzle-fill"></i> Activity 5: schema-Driven Test Data</span><br><br>
Call <code>StudentDataSchema.example(size=20)</code> to generate 20 synthetic rows. Confirm that the generated DataFrame passes <code>StudentDataSchema.validate()</code> without errors. Then confirm that if you corrupt one cell (set a score to 200), validation fails.
<pre>synthetic = StudentDataSchema.example(size=20)
StudentDataSchema.validate(synthetic)  # should pass</pre>
</div>

## Capstone: Data Contract for grade-predictor

Bring Pandera into the `grade-predictor` pipeline as a first-class data contract.

<div class='ark-activity'>
<span class='ark-activity-title'><i class="bi bi-puzzle-fill"></i> Capstone - End-to-End Validated Pipeline</span><br><br>
<ol>
<li>Define <code>StudentDataSchema</code> in <code>grade_predictor/schemas.py</code> covering all columns of <code>university_analytics.csv</code></li>
<li>Update <code>load_and_validate</code> (from Chapter 25) to run Pandera schema validation <em>after</em> row-level Pydantic validation</li>
<li>Add a <code>@pa.dataframe_check</code> that verifies the computed weighted average (using the weights from <code>PipelineConfig</code>) falls in [0, 100] for every row</li>
<li>Write three tests: one that the real CSV passes the schema, one that a DataFrame with a bad score raises <code>SchemaError</code>, and one that uses <code>example()</code> to generate synthetic data and confirms it passes</li>
<li>Run <code>uv run pytest -v</code> and confirm all three pass</li>
</ol>
</div>

## 6. Why Pandera? comparing schema validation tools

Before Pandera became the community standard for DataFrame validation, several tools tackled the same problem in different ways. Understanding the landscape helps you choose the right tool for your context.

<div class='ark-concept'>
<span class='ark-concept-title'><i class="bi bi-key-fill"></i> Key Concept: Schema validation is not the same as data cleaning</span><br><br>
Schema validation answers a yes/no question: does this DataFrame conform to the contract? It runs fast and fails loudly. Data cleaning transforms and repairs data. The two serve different purposes: validate at the pipeline boundary, clean before you get there.
</div>

| Tool | Best for | Limitation |
| --- | --- | --- |
| **Pandera** | DataFrame contracts in Python pipelines | pandas/polars-specific; not for arbitrary dicts |
| **Great Expectations** | Enterprise data quality suites, HTML reports, data docs | Heavy setup, complex configuration, overkill for most ML work |
| **Pydantic** | Row-level validation, API payloads, config models | Not designed for tabular data; looping over rows is slow |
| **Frictionless Data** | YAML-defined schemas, cross-language contracts | Less Python-native; validation logic lives in config files |
| **Cerberus / marshmallow** | Dict and object validation | No DataFrame support; designed for records not tables |

The practical split comes down to scope. Pydantic is the right choice when you are validating individual records at a system boundary: an API payload, a config file, a single row entering a pipeline. Pandera is the right choice when you are validating the structure and statistics of an entire DataFrame. They are complementary, not competing: Chapter 25 showed you Pydantic for the row, this notebook shows you Pandera for the table.

Great Expectations is powerful but designed for data engineering teams who need data documentation, alerting, and audit trails across multiple data sources. For most ML projects, Pandera gives you 90% of the value with 10% of the setup.

<div class='ark-tip'>
<span class='ark-tip-title'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Use both Pydantic and Pandera in the same pipeline</span><br><br>
Validate incoming JSON records with Pydantic as they arrive, then validate the assembled DataFrame with Pandera before it enters any model or transformation. The two checks run at different granularities and catch different classes of bugs: Pydantic catches a bad individual record, Pandera catches drift in the distribution across hundreds of records.
</div>

## Further reading

| Resource | Why it matters |
| --- | --- |
| [Pandera documentation](https://pandera.readthedocs.io/) | Full reference: DataFrameSchema, DataFrameModel, check types, Polars support |
| [Pandera + Polars](https://pandera.readthedocs.io/en/stable/polars.html) | Same API works on Polars DataFrames |
| [Pandera pytest integration](https://pandera.readthedocs.io/en/stable/pandas_on_spark.html) | `@pa.check_types` decorator for function-level schema enforcement |
| [Pydantic + Pandera together](https://pandera.readthedocs.io/en/stable/pydantic_integration.html) | Use a Pandera schema as a Pydantic field type |
| [Great Expectations](https://docs.greatexpectations.io/) | Enterprise-grade data quality platform; Pandera for code-first, GX for teams with data catalogs |

## Summary

| Concept | Key rule |
| --- | --- |
| `DataFrameSchema` | Functional API: describe columns, constraints, table-level checks as a dict |
| `DataFrameModel` | Class-based API: columns as annotated fields, checks as classmethods |
| `pa.Field(ge=, le=)` | Same constraint vocabulary as Pydantic's `Field` |
| `pa.Check.isin([...])` | Restrict a column to an explicit set of values |
| `@pa.check("col")` | Column-level custom check: receives the Series, returns bool or bool Series |
| `@pa.dataframe_check` | Cross-column check: receives the full DataFrame |
| `lazy=True` | Collect all failures; raises `SchemaErrors` (plural) |
| `lazy=False` | Stop on first failure; raises `SchemaError` |
| `schema.example(size=n)` | Generate n rows of valid synthetic data for testing |
| Pydantic + Pandera | Validate rows with Pydantic, validate the assembled table with Pandera |

**Next:** Chapter 27: Classical ML: Scikit-learn pipelines applied to the fully validated, typed `grade-predictor` dataset.